In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import itertools
import logging
from pathlib import Path
from typing import Any, TypeVar
import re

import h5py
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
import seaborn as sns
from tqdm.auto import tqdm

In [ ]:
from src.data import add_metadata_features
from src.viz import plot_epochs, add_timit_insets, add_pod_line, add_uv_annotation

In [ ]:
L = logging.getLogger(__name__)

In [ ]:
sns.set_context(font_scale=1.5)

In [ ]:
trf_eois = "outputs/trf_eois/eois.csv"
trf_stepwise_eois = "outputs/analyze_stepwise/eois.csv"

all_epoch_paths = list(Path("outputs/epochs_preprocessed").glob("*_epo.fif"))
epoched_phoneme_path = "/userdata/jgauthier/projects/ideal-word-representations/phoneme_epochs-all.h5"
epoched_phoneme_onset_path = "/userdata/jgauthier/projects/ideal-word-representations/phoneme_epochs-onsets.h5"
pval_threshold = 1e-3
uv_threshold = 0
outdir = "."

In [ ]:
timit_epoch_sources = {
    "All": epoched_phoneme_path,
    "Word onset": epoched_phoneme_onset_path,
}

In [ ]:
eoi_df = pd.read_csv(trf_eois, index_col=["subject", "electrode", "feature_block"])
eoi_df = eoi_df[eoi_df.unique_variance > uv_threshold]
eoi_df

In [ ]:
stepwise_eoi_df = pd.read_csv(trf_stepwise_eois, index_col=["subject", "electrode", "progression"])
stepwise_eoi_df = stepwise_eoi_df[stepwise_eoi_df.r2_diff > 0]

# We only care about the final step of each progression (testing gradient response)
stepwise_eoi_df = stepwise_eoi_df[stepwise_eoi_df.progression_step == 2]

stepwise_eoi_df = stepwise_eoi_df.sort_values("r2_diff", ascending=False)
stepwise_eoi_df

In [ ]:
epochs = {}
for path in tqdm(all_epoch_paths):
    subject_name = re.findall("(EC[\d]+)_epo", str(path))[0]
    epochs[subject_name] = mne.read_epochs(str(path), verbose=False)
    try:
        epochs[subject_name].metadata = add_metadata_features(epochs[subject_name].metadata)
    except AssertionError:
        del epochs[subject_name]
        continue

In [ ]:
all_md = pd.concat([epochs[subject_name].metadata for subject_name in epochs], names=["subject"], keys=epochs.keys())
g = sns.displot(all_md.behavior_linear)

# quantile cut
all_md["behavior_quantile"], bins = pd.cut(all_md.behavior_linear, 4, retbins=True)
for qleft, qright in zip(bins[:-1], bins[1:]):
    g.ax.axvline(qleft, color="black", linestyle="--")

In [ ]:
sns.lineplot(data=all_md, x="resampled", y="behavior_linear", hue="subject")

In [ ]:
all_md["correct"] = all_md.label_behavior == all_md.label_lexical
all_md.groupby("resampled").correct.mean()

In [ ]:
all_md["follows_acoustics"] = all_md.label_behavior == all_md.label_acoustic
all_md.groupby("resampled").follows_acoustics.mean()

## Plots

In [ ]:
epoch_times = next(iter(epochs.values())).times

In [ ]:
def make_epochs_df(feature_block, plot_eoi_df=None, baseline=None):
    all_plot_epochs, all_plot_epochs_df = {}, {}

    if plot_eoi_df is None:
        plot_eoi_df = eoi_df
    try:
        plot_eoi_df = plot_eoi_df.loc[(slice(None), slice(None), feature_block)]
    except KeyError:
        L.info(f"Feature block {feature_block} not found in eoi_df")
        return None, None

    epochs_ = epochs
    if baseline is not None:
        epochs_ = {subject: epochs[subject].copy().apply_baseline(baseline) for subject in epochs}

    for subject, channel in plot_eoi_df.index:
        plot_epochs = epochs_[subject]
        plot_epochs_df = plot_epochs.metadata.loc[plot_epochs.selection]
        plot_epochs_df["mismatch"] = plot_epochs_df.mismatch.map({-1: False, 1: True})
        plot_epochs_df["mismatch_left_right"] = plot_epochs_df["mismatch_left_right"].map({-1: "left", 1: "right"})
        plot_epochs_df["lexical_evidence_cue"] = plot_epochs_df["lexical_evidence_cue"].map({-1: "left", 1: "right"})
        plot_epochs_df["categorical_acoustic_cue"] = plot_epochs_df["categorical_acoustic_cue"].map({-1: "left", 1: "right"})
        plot_epochs_df["categorical_behavior"] = plot_epochs_df["behavior_categorical"].map({-1: "left", 0: "neither", 1: "right"})

        all_plot_epochs[subject, channel] = plot_epochs.get_data()[:, channel, :]
        all_plot_epochs_df[subject, channel] = plot_epochs_df

    all_plot_epochs_df = pd.concat(all_plot_epochs_df, names=["subject", "channel", "epoch_idx"])
    all_plot_epochs_df["facet_label"] = all_plot_epochs_df.index.get_level_values("subject").str.cat(
        (all_plot_epochs_df.index.get_level_values("channel") + 1).astype(str), sep="_")

    return all_plot_epochs, all_plot_epochs_df

In [ ]:
plot_specs = {
    ("eoi", "lexical_evidence"): {
        "": dict(baseline=None,
                hue="lexical_evidence_cue", hue_order=["left", "right"],
                share_groupers=True),
        "-baselined": dict(baseline=(None, 0),
                        hue="lexical_evidence_cue", hue_order=["left", "right"],
                        share_groupers=True),
        "-baselined-split": dict(baseline=(None, 0), hue="label_acoustic", style="label_lexical",
                                share_groupers=False),
        "-split": dict(baseline=None, hue="label_acoustic", style="label_lexical",
                       share_groupers=False),
        "-baselined-belief_update": dict(baseline=(None, 0),
                                         hue="belief_update_int", hue_order=[-5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5],
                                         palette="coolwarm",
                                         share_groupers=False),
        "-baselined-belief_update_coarse": dict(baseline=(None, 0),
                                         hue="belief_update_int_coarse", hue_order=[-5, -2, 0, 2, 5],
                                         palette="coolwarm",
                                         share_groupers=False),

        "-baselined-split-behavior": dict(baseline=(None, 0), hue="label_acoustic_emoji", style="label_behavior_emoji",
                                          share_groupers=False),
    },

    ("eoi", "acoustic"): {
        "": dict(baseline=None,
                hue="categorical_acoustic_cue", hue_order=["left", "right"],
                share_groupers=True),
        "-baselined": dict(baseline=(None, 0),
                        hue="categorical_acoustic_cue", hue_order=["left", "right"],
                        share_groupers=True),
        "-baselined-split": dict(baseline=(None, 0), hue="label_acoustic", style="label_lexical",
                                share_groupers=False),
        "-split": dict(baseline=None, hue="label_acoustic", style="label_lexical",
                       share_groupers=False),
        "-baselined-continuous": dict(baseline=(None, 0),
                                      hue="resampled", hue_order=[1, 2, 3, 4, 5, 6],
                                      palette="RdBu",
                                      share_groupers=True),

        "-baselined-split-behavior": dict(baseline=(None, 0), hue="label_acoustic_emoji", style="label_behavior_emoji",
                                          share_groupers=False),
    },

    ("eoi", "mismatch"): {
        "": dict(baseline=None,
                 hue="mismatch_left_right", hue_order=["left", "right"],
                 share_groupers=True),
        "-baselined": dict(baseline=(None, 0),
                           hue="mismatch_left_right", hue_order=["left", "right"],
                           share_groupers=True),
        "-baselined-split": dict(baseline=(None, 0), hue="label_acoustic", style="label_lexical",
                                 share_groupers=False),
        "-split": dict(baseline=None, hue="label_acoustic", style="label_lexical",
                       share_groupers=False),
        "-baselined-belief_update": dict(baseline=(None, 0),
                                         hue="belief_update_int", hue_order=[-5, -4, -3, 3, 4, 5],
                                         # NB hue order only includes belief updates which are consistent with a mismatch. other trials
                                         # will not be plotted
                                         palette="coolwarm",
                                         share_groupers=True),
        "-baselined-belief_update_coarse": dict(baseline=(None, 0),
                                         hue="belief_update_int_coarse_mismatch_only", hue_order=[-5, -2, 0, 2, 5],
                                         palette="coolwarm",
                                         share_groupers=True),

        "-baselined-split-behavior": dict(baseline=(None, 0), hue="label_acoustic_emoji", style="label_behavior_emoji",
                                          share_groupers=False),
    },

    ("stepwise_eoi", "acoustic"): {
        "-baselined-continuous": dict(baseline=(None, 0),
                                      hue="resampled", hue_order=[1, 2, 3, 4, 5, 6],
                                      palette="coolwarm",
                                      share_groupers=True),
    },

    ("stepwise_eoi", "mismatch"): {
        "-baselined-belief_update": dict(baseline=(None, 0),
                                         hue="belief_update_int", hue_order=[-5, -4, -3, 3, 4, 5],
                                         # NB hue order only includes belief updates which are consistent with a mismatch. other trials
                                         # will not be plotted
                                         palette="coolwarm",
                                         share_groupers=True),
    },
}

In [ ]:
# # For debugging

feature_block = "acoustic"
suffix = "-baselined-continuous"
feature_block = "mismatch"
suffix = "-baselined-split-behavior"
kwargs = plot_specs[("eoi", feature_block)][suffix]

ep, ep_df = make_epochs_df(feature_block, plot_eoi_df=eoi_df.loc[["EC260"]].loc[(slice(None), [92, 219]), :],
                           baseline=kwargs.get("baseline"))
plot_kwargs = {k: v for k, v in kwargs.items() if k not in ["baseline"]}
plot_kwargs["smoke_test"] = True
# plot_kwargs["errorbar"] = None
g, p_ep, p_df = plot_epochs(ep, ep_df, epoch_times=epoch_times, **plot_kwargs)
g = add_timit_insets(g, timit_epoch_sources)

In [ ]:
for (eoi_source, feature_block), plot_spec in tqdm(plot_specs.items()):
    for suffix, kwargs in tqdm(plot_spec.items(), leave=False):
        make_kwarg_names = ["baseline"]

        make_kwargs = {k: v for k, v in kwargs.items() if k in make_kwarg_names}
        plot_kwargs = {k: v for k, v in kwargs.items() if k not in make_kwarg_names}

        if eoi_source == "eoi":
            plot_eoi_df = eoi_df
        elif eoi_source == "stepwise_eoi":
            plot_eoi_df = stepwise_eoi_df
        else:
            raise ValueError(f"Unknown eoi_source {eoi_source}")
        ep, ep_df = make_epochs_df(feature_block, plot_eoi_df=plot_eoi_df, **make_kwargs)

        g, p_ep, p_df = plot_epochs(ep, ep_df, close=False,
                                    epoch_times=epoch_times,
                                    **plot_kwargs)
        g = add_uv_annotation(g, feature_block, eoi_df)
        g = add_pod_line(g)
        g = add_timit_insets(g, timit_epoch_sources)
        g.savefig(f"{outdir}/epochs-{eoi_source}-{feature_block}{suffix}.pdf")
        plt.close(g.figure)